# 07 — Technology Alternatives to PSTN

Analyzes PSTN-alternative technologies (cable/web vs wireless/non-web), enriches the solution registry with technical metadata, and generates visualizations for protocol complexity, regional adoption, and cost-complexity tradeoffs.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.research.tech_researcher import (
    generate_awesome_list,
    enrich_registry,
    score_protocol_richness,
    score_security_posture,
)
from src.research.product_researcher import analyze_registry

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# Generate awesome list (PSTN alternatives catalog)
awesome = generate_awesome_list()
print(f"Awesome list loaded: {len(awesome)} alternatives")

# Load solution registry
reg_path = Path("data/processed/solution_registry.csv")
if reg_path.exists():
    registry = pd.read_csv(reg_path)
    print(f"Registry loaded from CSV: {len(registry)} solutions")
else:
    print("CSV not found, re-running analyze_registry...")
    registry = analyze_registry()
    print(f"Registry generated: {len(registry)} solutions")

display(registry.head(3))

In [ ]:
# Enrich registry with technical metadata
enriched = enrich_registry(registry)
print(f"Enriched registry: {len(enriched)} rows")
print(f"Columns added: protocol_richness, security_score")
display(enriched[["name", "vendor", "protocol_richness", "security_score"]].head())

In [ ]:
# --- 4a: PSTN alternatives by category and medium ---
fig, ax = plt.subplots(figsize=(10, 5))
ct = pd.crosstab(awesome["category"], awesome["medium"])
ct.plot(kind="bar", ax=ax, width=0.7)
ax.set_title("PSTN Alternatives by Category and Medium", fontsize=14, fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Count")
ax.legend(title="Medium", bbox_to_anchor=(1.05, 1), loc="upper left")
fig.tight_layout()
fig.savefig("data/processed/pstn_alternatives_by_category.png", dpi=150)
plt.show()

# --- 4b: Protocol complexity scatter ---
if "protocol_richness" in enriched.columns and "security_score" in enriched.columns:
    fig, ax = plt.subplots(figsize=(9, 6))
    scatter = ax.scatter(
        enriched["protocol_richness"],
        enriched["security_score"],
        c=enriched["security_score"],
        cmap="viridis",
        s=80,
        alpha=0.75,
        edgecolors="k",
        linewidth=0.5,
    )
    for _, row in enriched.iterrows():
        ax.annotate(
            row["vendor"][:12],
            (row["protocol_richness"], row["security_score"]),
            fontsize=6,
            alpha=0.8,
            xytext=(4, 4),
            textcoords="offset points",
        )
    ax.set_title("Protocol Complexity vs Security Score", fontsize=14, fontweight="bold")
    ax.set_xlabel("Protocol Richness (number of tech capabilities matched)")
    ax.set_ylabel("Security Score (0-10)")
    fig.colorbar(scatter, ax=ax, label="Security Score")
    fig.tight_layout()
    fig.savefig("data/processed/protocol_complexity.png", dpi=150)
    plt.show()
else:
    print("Skipping protocol_complexity — missing columns in enriched registry")

# --- 4c: Regional tech adoption heatmap ---
if "continent" in enriched.columns and "lifecycle_assigned" in enriched.columns:
    pivot = pd.crosstab(enriched["continent"], enriched["lifecycle_assigned"])
    cat_order = [c for c in ["cutting_edge", "mature_active", "most_used_current", "most_used_eol"] if c in pivot.columns]
    pivot = pivot[cat_order]

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title("Regional Technology Adoption — Lifecycle per Continent", fontsize=14, fontweight="bold")
    ax.set_xlabel("Lifecycle Category")
    ax.set_ylabel("Continent")
    fig.tight_layout()
    fig.savefig("data/processed/regional_tech_adoption.png", dpi=150)
    plt.show()
else:
    print("Skipping regional_tech_adoption — missing continent/lifecycle columns")

# --- 4d: Cost vs complexity tradeoff ---
cost_map = {"Very Low": 1, "Low": 2, "Low-Medium": 3, "Medium": 4, "Medium-High": 5, "High": 6}
complexity_map = {"Very Low": 1, "Low": 2, "Low-Medium": 3, "Medium": 4, "Medium-High": 5, "High": 6, "Very High": 7}

if "cost" in awesome.columns and "complexity" in awesome.columns:
    fig, ax = plt.subplots(figsize=(9, 6))
    awesome_plot = awesome.copy()
    awesome_plot["cost_num"] = awesome_plot["cost"].map(cost_map)
    awesome_plot["complexity_num"] = awesome_plot["complexity"].map(complexity_map)
    awesome_plot = awesome_plot.dropna(subset=["cost_num", "complexity_num"])

    palette = {"web": "#3498db", "non_web": "#e74c3c"}
    for cat in ["web", "non_web"]:
        subset = awesome_plot[awesome_plot["category"] == cat]
        ax.scatter(
            subset["cost_num"],
            subset["complexity_num"],
            c=palette.get(cat, "#888"),
            label=cat,
            s=100,
            alpha=0.8,
            edgecolors="k",
            linewidth=0.5,
        )
        for _, row in subset.iterrows():
            ax.annotate(
                row["name"][:18],
                (row["cost_num"], row["complexity_num"]),
                fontsize=7,
                alpha=0.8,
                xytext=(4, 4),
                textcoords="offset points",
            )

    ax.set_title("Cost vs Complexity Tradeoff for PSTN Alternatives", fontsize=14, fontweight="bold")
    ax.set_xlabel("Cost (1=Very Low, 6=High)")
    ax.set_ylabel("Complexity (1=Very Low, 7=Very High)")
    ax.set_xticks(list(cost_map.values()))
    ax.set_xticklabels(list(cost_map.keys()), rotation=45, ha="right")
    ax.set_yticks(list(complexity_map.values()))
    ax.set_yticklabels(list(complexity_map.keys()))
    ax.legend(title="Category")
    fig.tight_layout()
    fig.savefig("data/processed/cost_complexity_tradeoff.png", dpi=150)
    plt.show()
else:
    print("Skipping cost_complexity_tradeoff — missing cost/complexity columns")

In [ ]:
# Display awesome list as formatted table
display(awesome.style.set_table_styles([
    {"selector": "thead th", "props": [("text-align", "left"), ("font-weight", "bold")]},
    {"selector": "tbody td", "props": [("text-align", "left")]},
]).set_properties(**{"max-width": "300px", "white-space": "normal"}))

In [ ]:
# Key statistics
total = len(awesome)
cable_count = len(awesome[awesome["category"] == "web"])
wireless_count = len(awesome[awesome["category"] == "non_web"])

if "security_score" in enriched.columns:
    avg_security = enriched["security_score"].mean()
else:
    avg_security = None

if "protocol_richness" in enriched.columns:
    avg_protocol = enriched["protocol_richness"].mean()
else:
    avg_protocol = None

print("=" * 60)
print("PSTN Alternatives — Key Statistics")
print("=" * 60)
print(f"  Total alternatives catalogued:  {total}")
print(f"  Cable/web-based alternatives:  {cable_count}")
print(f"  Wireless/non-web alternatives: {wireless_count}")
if avg_security is not None:
    print(f"  Avg security score (registry): {avg_security:.2f} / 10")
if avg_protocol is not None:
    print(f"  Avg protocol richness (registry): {avg_protocol:.2f}")
print("=" * 60)

In [ ]:
# Save enriched registry
enriched.to_csv("data/processed/tech_enriched_registry.csv", index=False)
print(f"Enriched registry saved: data/processed/tech_enriched_registry.csv ({len(enriched)} rows)")

# Save awesome list
awesome.to_csv("data/processed/awesome_list.csv", index=False)
print(f"Awesome list saved: data/processed/awesome_list.csv ({len(awesome)} rows)")

**EN — Alternatives catalog.** Each alternative is tagged with a transport `medium` (e.g. ethernet_ip, electrical_contact, cellular). The RAG retriever uses this to honour negative constraints — if a request says 'no ethernet / no analog phone line', options on those media are excluded before ranking. Match recommended device scale and cost to your deployment size.

**繁中 — 替代方案目錄。** 每個替代方案標註傳輸 `medium`（如 ethernet_ip、electrical_contact、cellular）。RAG 檢索器據此遵守負向限制——若需求為「不可用乙太網路／類比電話線」，採用該媒介的方案會在排序前被排除。請依部署規模對應建議裝置數與成本。

## 7.5 Null-Hypothesis Test — Cybersecurity & integration feasibility of replacing the physical PBX

**EN.** This is the **cybersecurity + integration** leg of the four-part feasibility frame
(financial → NB 10, technical/lifecycle → NB 06, **cybersecurity & integration → here**). We ask
whether the modern PSTN alternatives are a *materially better and feasibly integrable* security
posture than a kept physical PBX — i.e. whether replacement is justified on **security necessity**
**and** is **technically feasible to integrate** alongside / in place of the existing physical PBX.

> *Do PSTN alternatives deliver a security posture above the physical-PBX baseline, while remaining
> feasible to integrate (bounded complexity)?*

We run **two coupled one-sided tests** (joint decision = reject only if **both** reject):

- **(A) Security necessity** — `security_score` (0–10) scored from each alternative's catalog
  `security` / `pros` text (TLS, SRTP, AES, mTLS, MFA, OAuth, end-to-end).
  - **H₀ₐ:** alternatives are **not** a security upgrade — mean `security_score ≤ 5.0` (the neutral
    physical-PBX baseline).
  - **H₁ₐ:** mean `security_score > 5.0`.
  - **Test:** one-sample, **one-sided t-test**; report **Cohen's d** and **95% CI**.
- **(B) Integration feasibility** — `complexity` of the alternatives catalog (mapped 1–7).
  - **H₀_b:** integration is **not** feasible at scale — mean complexity **≥ 5** (Medium-High+).
  - **H₁_b:** mean complexity **< 5** (integrates within bounded human-time cost).
  - **Test:** one-sample, **one-sided t-test**.

**Decision rule:** reject the **joint H₀** at **α = 0.05** only if **both** A and B reject — i.e.
replacement is a *security necessity* **and** *integration-feasible*. This is disjoint from NB 10
(financial NPV) and NB 06 (lifecycle obsolescence): no metric is shared.

**繁中.** 這是四維可行性框架中的**資安＋整合**面向（財務→NB 10，技術／生命週期→NB 06，
**資安與整合→此處**）。檢視現代 PSTN 替代方案是否在資安上**顯著優於**保留實體 PBX，且
**可行地整合**至／取代既有實體 PBX。採**兩個耦合單尾檢定**（聯合判定：兩者皆拒絕才拒絕）：

- **(A) 資安必要性：** `security_score`（0–10）由各替代方案目錄的 `security`／`pros` 文字評分
  （TLS、SRTP、AES、mTLS、MFA、OAuth、端對端）。H₀ₐ：≤ 5.0（中性基準）；H₁ₐ：> 5.0。單尾 t 檢定，報告 Cohen's d 與 95% CI。
- **(B) 整合可行性：** H₀_b：平均複雜度 ≥ 5（偏高）；H₁_b：< 5（人力時間成本可控）。單尾 t 檢定。
- **判定：** α = 0.05；A、B 皆拒絕方拒絕**聯合 H₀**。與 NB 10（財務）、NB 06（生命週期）不重疊。

In [ ]:
from scipy import stats as _stats

ALPHA = 0.05
SEC_BASELINE = 5.0       # neutral physical-PBX security baseline (0-10)
COMPLEXITY_FEASIBLE = 5  # mean complexity must be < 5 (below Medium-High) to be integration-feasible

def _one_sample_t(sample, popmean, alternative):
    sample = np.asarray(sample, dtype=float)
    sample = sample[~np.isnan(sample)]
    n = sample.size
    res = _stats.ttest_1samp(sample, popmean, alternative=alternative)
    sd = sample.std(ddof=1)
    cohens_d = (sample.mean() - popmean) / sd if sd else float('nan')
    se = sd / np.sqrt(n) if n else float('nan')
    tcrit = _stats.t.ppf(1 - ALPHA, df=n - 1) if n > 1 else float('nan')
    return {
        'n': n, 'mean': float(sample.mean()) if n else float('nan'), 'popmean': popmean,
        't_stat': res.statistic, 'p_value': res.pvalue,
        'cohens_d': cohens_d,
        'ci_low': sample.mean() - tcrit * se, 'ci_high': sample.mean() + tcrit * se,
        'reject_H0': bool(res.pvalue < ALPHA),
    }

# Security posture comes from the alternatives catalog's free-text `security`/`pros`
# columns (the enriched-registry security_score is keyword-empty). Reuse the same
# keyword scorer used elsewhere for consistency.
def _row_security_score(row):
    sec = row.get("security", "") or ""
    pros = row.get("pros", "")
    pros_list = pros.split(";") if isinstance(pros, str) else list(pros) if pros else []
    return score_security_posture({"tags": [str(sec)], "pros": [str(p) for p in pros_list]})

awesome_sec = awesome.apply(_row_security_score, axis=1)

# --- (A) Security necessity: H1 mean security_score > 5.0 ---
A = _one_sample_t(awesome_sec, SEC_BASELINE, alternative='greater')

# --- (B) Integration feasibility: H1 mean complexity < 5 ---
complexity_map = {"Very Low": 1, "Low": 2, "Low-Medium": 3, "Medium": 4,
                  "Medium-High": 5, "High": 6, "Very High": 7}
complexity_num = awesome['complexity'].map(complexity_map)
B = _one_sample_t(complexity_num, COMPLEXITY_FEASIBLE, alternative='less')

joint_reject = bool(A['reject_H0'] and B['reject_H0'])

print('=' * 64)
print('  7.5  CYBERSECURITY & INTEGRATION — Null-Hypothesis Test')
print('=' * 64)
print('(A) Security necessity   H0: mean security_score <= 5.0   H1: > 5.0')
print(f"    n={A['n']}  mean={A['mean']:.2f}  t={A['t_stat']:.3f}  p={A['p_value']:.4g}")
print(f"    Cohen's d={A['cohens_d']:.3f}  95% CI=[{A['ci_low']:.2f}, {A['ci_high']:.2f}]  "
      f"reject_H0={A['reject_H0']}")
print('-' * 64)
print('(B) Integration feasible H0: mean complexity >= 5      H1: < 5')
print(f"    n={B['n']}  mean={B['mean']:.2f}  t={B['t_stat']:.3f}  p={B['p_value']:.4g}")
print(f"    Cohen's d={B['cohens_d']:.3f}  95% CI=[{B['ci_low']:.2f}, {B['ci_high']:.2f}]  "
      f"reject_H0={B['reject_H0']}")
print('=' * 64)
if joint_reject:
    print("  JOINT REJECT H0: replacing the physical PBX is a SECURITY")
    print("  NECESSITY *and* is INTEGRATION-FEASIBLE (bounded complexity).")
else:
    print("  FAIL TO JOINTLY REJECT H0: the cybersecurity+integration case")
    print("  for replacement is NOT established (need both A and B to reject).")
print('=' * 64)

necessity_security = pd.DataFrame([
    {'dimension': 'cybersecurity', 'metric': 'security_score', 'alternative': 'greater',
     'popmean': SEC_BASELINE, **{k: A[k] for k in
        ('n', 'mean', 't_stat', 'p_value', 'cohens_d', 'ci_low', 'ci_high', 'reject_H0')}},
    {'dimension': 'integration', 'metric': 'complexity', 'alternative': 'less',
     'popmean': COMPLEXITY_FEASIBLE, **{k: B[k] for k in
        ('n', 'mean', 't_stat', 'p_value', 'cohens_d', 'ci_low', 'ci_high', 'reject_H0')}},
])
necessity_security['alpha'] = ALPHA
necessity_security['joint_reject_H0'] = joint_reject
necessity_security['verdict'] = (
    'replacement_security_integration_justified' if joint_reject
    else 'not_established'
)
_out = Path('data/processed/replacement_necessity_security.csv')
necessity_security.to_csv(_out, index=False)
print(f"Saved → {_out.resolve()}")
display(necessity_security)